# Matching Experiments con preferenze di razza

Questo notebook permette di testare il sistema di matching cane–famiglia su più profili adottanti.

Rispetto alla versione precedente, il sistema considera anche le **razze preferite** indicate nel file `family_profiles.json`.

Per ogni famiglia il sistema calcola:

- compatibilità strutturata;
- punteggio di razza (`breed_score`);
- similarità semantica tramite BERT;
- score finale;
- classifica dei cani consigliati.


## 1. Import delle librerie

In [1]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

## 2. Caricamento dei dati

Vengono caricati:

- `dogs_clean.csv`;
- `bert_embeddings.npy`;
- `family_profiles.json`.

Se `dogs_clean.csv` non contiene ancora le colonne `breed1_label` e `breed2_label`, vengono create automaticamente tramite `breed_labels.csv`.


In [2]:
dogs = pd.read_csv("../data/processed/dogs_clean.csv")
bert_embeddings = np.load("../data/processed/bert_embeddings.npy")

# Creazione automatica delle etichette di razza se non già presenti
if "breed1_label" not in dogs.columns or "breed2_label" not in dogs.columns:
    breed_labels = pd.read_csv("../data/raw/breed_labels.csv")

    breed_map = dict(
        zip(
            breed_labels["BreedID"],
            breed_labels["BreedName"]
        )
    )

    dogs["breed1_label"] = dogs["Breed1"].map(breed_map)
    dogs["breed2_label"] = dogs["Breed2"].map(breed_map)
    dogs["breed2_label"] = dogs["breed2_label"].fillna("None")

profiles_path = Path("../data/family_profiles.json")

with open(profiles_path, "r", encoding="utf-8") as f:
    family_profiles = json.load(f)

print("Dataset cani:", dogs.shape)
print("Embedding BERT:", bert_embeddings.shape)
print("Profili famiglia disponibili:")
for profile_id in family_profiles.keys():
    print("-", profile_id)

Dataset cani: (8132, 32)
Embedding BERT: (8132, 768)
Profili famiglia disponibili:
- family_children_apartment
- single_active_person
- senior_applicant
- experienced_countryside_family
- first_time_adopter


## 3. Scelta del profilo famiglia

Modifica solo `selected_profile_id` per scegliere su quale famiglia fare la prova.

Esempi:

- `family_children_apartment`
- `single_active_person`
- `senior_applicant`
- `experienced_countryside_family`
- `first_time_adopter`

Ogni profilo può contenere anche il campo:

```python
"preferred_breeds": ["Labrador Retriever", "Golden Retriever"]
```

Se la lista è vuota, significa che la famiglia non ha preferenze di razza.


In [3]:
selected_profile_id = "family_children_apartment"

family_profile = family_profiles[selected_profile_id]

family_profile

{'profile_name': 'Famiglia con bambini in appartamento',
 'applicant_age': '31-60',
 'children': True,
 'dog_experience': 'some',
 'housing': 'apartment',
 'garden': False,
 'preferred_gender': 'indifferent',
 'preferred_age': 'young',
 'preferred_size': 'small',
 'preferred_fur': 'short',
 'daily_time': '2-4',
 'activity_level': 'moderate',
 'preferred_breeds': ['Labrador Retriever', 'Golden Retriever'],
 'ideal_dog_description': 'We are looking for a small, affectionate and friendly dog suitable for a family with children. The dog should be calm at home, sociable, easy to manage and suitable for apartment life.'}

## 4. Funzioni di compatibilità

La compatibilità strutturata viene calcolata con una media pesata.

La razza ha peso maggiore rispetto ad altre preferenze:

- razza principale corrispondente (`breed1_label`) → `breed_score = 1.0`;
- seconda razza corrispondente (`breed2_label`) → `breed_score = 0.6`;
- nessuna corrispondenza → `breed_score = 0.0`;
- nessuna preferenza di razza → `breed_score = 1.0`.

Nel calcolo complessivo, `breed_score` ha peso 3.


In [4]:
def normalize_text(text):
    return str(text).strip().lower()


def match_preference(preference, dog_value):
    if preference == "indifferent":
        return 1.0
    return 1.0 if preference == dog_value else 0.0


def size_similarity(preferred_size, dog_size):
    if preferred_size == "indifferent":
        return 1.0

    size_order = {
        "small": 0,
        "medium": 1,
        "large": 2,
        "extra_large": 3
    }

    if preferred_size not in size_order or dog_size not in size_order:
        return 0.0

    distance = abs(size_order[preferred_size] - size_order[dog_size])

    if distance == 0:
        return 1.0
    elif distance == 1:
        return 0.5
    else:
        return 0.0


def compute_breed_score(dog, family):
    preferred_breeds = family.get("preferred_breeds", [])

    if len(preferred_breeds) == 0:
        return 1.0

    preferred_breeds = [
        normalize_text(breed)
        for breed in preferred_breeds
    ]

    breed1 = normalize_text(dog.get("breed1_label", ""))
    breed2 = normalize_text(dog.get("breed2_label", ""))

    if breed1 in preferred_breeds:
        return 1.0

    if breed2 in preferred_breeds:
        return 0.6

    return 0.0


def compute_structured_score(dog, family):
    # Vincolo hard: meno di 1 ora al giorno non è compatibile con l'adozione.
    if family["daily_time"] == "<1":
        return 0.0

    # Vincolo hard: cani extra large richiedono un giardino.
    if dog["maturity_size_label"] == "extra_large" and not family["garden"]:
        return 0.0

    # Vincolo hard: nessuna esperienza + cane extra large.
    if family["dog_experience"] == "none" and dog["maturity_size_label"] == "extra_large":
        return 0.0

    weighted_scores = []

    gender_score = match_preference(family["preferred_gender"], dog["gender_label"])
    age_score = match_preference(family["preferred_age"], dog["age_group"])
    size_score = size_similarity(family["preferred_size"], dog["maturity_size_label"])
    fur_score = match_preference(family["preferred_fur"], dog["fur_length_label"])
    breed_score = compute_breed_score(dog, family)

    weighted_scores.append((gender_score, 1))
    weighted_scores.append((age_score, 1))
    weighted_scores.append((size_score, 1))
    weighted_scores.append((fur_score, 1))

    # La razza pesa di più
    weighted_scores.append((breed_score, 3))

    # Regola bambini
    if family["children"]:
        children_score = 1.0 if dog["age_group"] in ["puppy", "young"] else 0.5
        weighted_scores.append((children_score, 1))

    # Regola giardino e taglia
    if dog["maturity_size_label"] == "large":
        garden_score = 1.0 if family["garden"] else 0.2
        weighted_scores.append((garden_score, 1))
    elif dog["maturity_size_label"] == "extra_large":
        weighted_scores.append((1.0, 1))
    else:
        weighted_scores.append((1.0, 1))

    # Regola tempo disponibile
    if family["daily_time"] == "1-2":
        time_score = 0.6 if dog["age_group"] == "puppy" else 0.8
        weighted_scores.append((time_score, 1))
    else:
        weighted_scores.append((1.0, 1))

    # Regola esperienza
    if family["dog_experience"] == "none":
        if dog["maturity_size_label"] == "large":
            experience_score = 0.2
        elif dog["age_group"] == "puppy":
            experience_score = 0.5
        else:
            experience_score = 1.0
    else:
        experience_score = 1.0

    weighted_scores.append((experience_score, 1))

    # Regola over 60
    if family["applicant_age"] == "over_60":
        senior_score = 1.0 if dog["age_group"] in ["adult", "senior"] else 0.4
        weighted_scores.append((senior_score, 1))

    total_score = sum(score * weight for score, weight in weighted_scores)
    total_weight = sum(weight for score, weight in weighted_scores)

    return total_score / total_weight

## 5. Calcolo dello score strutturato per la famiglia scelta

In [5]:
dogs_experiment = dogs.copy()

dogs_experiment["breed_score"] = dogs_experiment.apply(
    lambda row: compute_breed_score(row, family_profile),
    axis=1
)

dogs_experiment["structured_score"] = dogs_experiment.apply(
    lambda row: compute_structured_score(row, family_profile),
    axis=1
)

print("Numero totale cani:", len(dogs_experiment))
print("Cani esclusi dai vincoli hard:", (dogs_experiment["structured_score"] == 0).sum())
print("Cani valutati:", (dogs_experiment["structured_score"] > 0).sum())

dogs_experiment[[
    "Name",
    "breed1_label",
    "breed2_label",
    "breed_score",
    "age_group",
    "maturity_size_label",
    "structured_score"
]].head()

Numero totale cani: 8132
Cani esclusi dai vincoli hard: 22
Cani valutati: 8110


,Name,breed1_label,breed2_label,breed_score,age_group,maturity_size_label,structured_score
0,Brisco,Mixed Breed,NaN,0.0,puppy,medium,0.500000
1,Miko,Mixed Breed,NaN,0.0,puppy,medium,0.590909
2,Hunter,Mixed Breed,NaN,0.0,puppy,medium,0.590909
3,Siu Pak & Her 6 Puppies,Mixed Breed,NaN,0.0,puppy,medium,0.590909
4,Bear,Mixed Breed,NaN,0.0,puppy,medium,0.590909


## 6. Similarità semantica tramite BERT

La descrizione del cane ideale della famiglia viene confrontata con le descrizioni dei cani tramite embedding BERT e cosine similarity.


In [6]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)
bert_model.eval()

print("BERT model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model loaded.


In [7]:
def get_bert_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = bert_model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.squeeze().numpy()


family_embedding = get_bert_embedding(family_profile["ideal_dog_description"])

semantic_scores = cosine_similarity(
    family_embedding.reshape(1, -1),
    bert_embeddings
).flatten()

dogs_experiment["bert_similarity"] = (semantic_scores + 1) / 2

dogs_experiment[["Name", "bert_similarity"]].head()

,Name,bert_similarity
0,Brisco,0.907398
1,Miko,0.876957
2,Hunter,0.925550
3,Siu Pak & Her 6 Puppies,0.822069
4,Bear,0.827598


## 7. Score finale

Lo score finale combina:

```text
final_score = 0.7 * structured_score + 0.3 * bert_similarity
```

La razza contribuisce allo `structured_score`, quindi influenza anche lo score finale.


In [8]:
dogs_experiment["final_score"] = (
    0.7 * dogs_experiment["structured_score"] +
    0.3 * dogs_experiment["bert_similarity"]
)

dogs_experiment[[
    "Name",
    "breed_score",
    "structured_score",
    "bert_similarity",
    "final_score"
]].head()

,Name,breed_score,structured_score,bert_similarity,final_score
0,Brisco,0.0,0.500000,0.907398,0.622220
1,Miko,0.0,0.590909,0.876957,0.676724
2,Hunter,0.0,0.590909,0.925550,0.691301
3,Siu Pak & Her 6 Puppies,0.0,0.590909,0.822069,0.660257
4,Bear,0.0,0.590909,0.827598,0.661916


## 8. Classifica finale per la famiglia scelta

In [9]:
ranking = dogs_experiment.sort_values(
    by="final_score",
    ascending=False
)[[
    "Name",
    "Age",
    "breed1_label",
    "breed2_label",
    "breed_score",
    "age_group",
    "gender_label",
    "maturity_size_label",
    "fur_length_label",
    "PhotoAmt",
    "structured_score",
    "bert_similarity",
    "final_score",
    "Description"
]].head(10)

ranking

,Name,Age,breed1_label,breed2_label,breed_score,age_group,gender_label,maturity_size_label,fur_length_label,PhotoAmt,structured_score,bert_similarity,final_score,Description
5743,Bobby,8,Golden Retriever,NaN,1.0,young,male,small,short,5.0,1.000000,0.884814,0.965444,Golden retriever 8months old.Need a loving hom...
8089,Pacino,12,Labrador Retriever,Mixed Breed,1.0,young,male,medium,short,7.0,0.954545,0.911148,0.941526,PAcino was found at a park in Damansara Kim on...
7580,"Mummy Moon, Luna And Ariel",15,Golden Retriever,Labrador Retriever,1.0,young,female,medium,short,3.0,0.954545,0.902593,0.938960,Three pups were rescued during Thaipusam and f...
3697,2 Black Dude,8,Labrador Retriever,Mixed Breed,1.0,young,mixed,medium,short,7.0,0.954545,0.901899,0.938751,"siblings brother and sister, big paws and shin..."
2549,FLUFFY,22,Golden Retriever,NaN,1.0,young,female,medium,short,2.0,0.954545,0.899024,0.937889,i am the owner for this dog and i am looking f...
1285,Whiskey,12,Labrador Retriever,Mixed Breed,1.0,young,male,medium,short,2.0,0.954545,0.897704,0.937493,Whiskey is our own pet. An offspring of my aun...
1898,Nil,12,Labrador Retriever,Mixed Breed,1.0,young,male,medium,short,1.0,0.954545,0.864164,0.927431,I'm a labrador retriever cross local. I have t...
3309,Snow,12,Labrador Retriever,Mixed Breed,1.0,young,female,small,long,4.0,0.909091,0.913736,0.910484,Snow used to have a home nearby my area but wh...
37,Baby / Bubbles,2,Labrador Retriever,Jack Russell Terrier,1.0,puppy,male,small,short,5.0,0.909091,0.894526,0.904721,3 month-old male puppy for adoption. a cross b...
2757,Simba,2,Labrador Retriever,Beagle,1.0,puppy,female,small,short,0.0,0.909091,0.838549,0.887928,Simba is very loving. She is also very playful...


## 9. Spiegazione automatica della raccomandazione

In [10]:
def explain_recommendation(dog, family, all_dogs):
    reasons = []

    if family["preferred_age"] == "indifferent" or dog["age_group"] == family["preferred_age"]:
        reasons.append("fascia d'età compatibile")

    if family["preferred_size"] == "indifferent" or dog["maturity_size_label"] == family["preferred_size"]:
        reasons.append("taglia compatibile")

    if family["preferred_gender"] == "indifferent" or dog["gender_label"] == family["preferred_gender"]:
        reasons.append("sesso compatibile")

    if family["preferred_fur"] == "indifferent" or dog["fur_length_label"] == family["preferred_fur"]:
        reasons.append("lunghezza del pelo compatibile")

    if len(family.get("preferred_breeds", [])) > 0:
        breed_score = compute_breed_score(dog, family)

        if breed_score == 1.0:
            reasons.append("razza principale compatibile")
        elif breed_score == 0.6:
            reasons.append("seconda razza compatibile")

    if dog["bert_similarity"] > all_dogs["bert_similarity"].quantile(0.75):
        reasons.append("descrizione semanticamente vicina al cane ideale")

    if dog["structured_score"] == 1.0:
        reasons.append("profilo strutturale altamente compatibile")

    if len(reasons) == 0:
        return "Compatibilità basata sul punteggio complessivo."

    return ", ".join(reasons)


ranking_with_explanations = ranking.copy()

ranking_with_explanations["explanation"] = ranking_with_explanations.apply(
    lambda row: explain_recommendation(row, family_profile, dogs_experiment),
    axis=1
)

ranking_with_explanations[[
    "Name",
    "breed1_label",
    "breed2_label",
    "breed_score",
    "final_score",
    "explanation"
]]

,Name,breed1_label,breed2_label,breed_score,final_score,explanation
5743,Bobby,Golden Retriever,NaN,1.0,0.965444,"fascia d'età compatibile, taglia compatibile, ..."
8089,Pacino,Labrador Retriever,Mixed Breed,1.0,0.941526,"fascia d'età compatibile, sesso compatibile, l..."
7580,"Mummy Moon, Luna And Ariel",Golden Retriever,Labrador Retriever,1.0,0.938960,"fascia d'età compatibile, sesso compatibile, l..."
3697,2 Black Dude,Labrador Retriever,Mixed Breed,1.0,0.938751,"fascia d'età compatibile, sesso compatibile, l..."
2549,FLUFFY,Golden Retriever,NaN,1.0,0.937889,"fascia d'età compatibile, sesso compatibile, l..."
1285,Whiskey,Labrador Retriever,Mixed Breed,1.0,0.937493,"fascia d'età compatibile, sesso compatibile, l..."
1898,Nil,Labrador Retriever,Mixed Breed,1.0,0.927431,"fascia d'età compatibile, sesso compatibile, l..."
3309,Snow,Labrador Retriever,Mixed Breed,1.0,0.910484,"fascia d'età compatibile, taglia compatibile, ..."
37,Baby / Bubbles,Labrador Retriever,Jack Russell Terrier,1.0,0.904721,"taglia compatibile, sesso compatibile, lunghez..."
2757,Simba,Labrador Retriever,Beagle,1.0,0.887928,"taglia compatibile, sesso compatibile, lunghez..."


## 10. Confronto rapido tra profili famiglia

Questa cella calcola il miglior cane consigliato per ogni profilo famiglia disponibile.

Serve per verificare che famiglie diverse producano ranking diversi anche in base alle preferenze di razza.


In [11]:
summary_rows = []

for profile_id, profile in family_profiles.items():
    temp = dogs.copy()

    temp["breed_score"] = temp.apply(
        lambda row: compute_breed_score(row, profile),
        axis=1
    )

    temp["structured_score"] = temp.apply(
        lambda row: compute_structured_score(row, profile),
        axis=1
    )

    profile_embedding = get_bert_embedding(profile["ideal_dog_description"])

    profile_semantic_scores = cosine_similarity(
        profile_embedding.reshape(1, -1),
        bert_embeddings
    ).flatten()

    temp["bert_similarity"] = (profile_semantic_scores + 1) / 2

    temp["final_score"] = (
        0.7 * temp["structured_score"] +
        0.3 * temp["bert_similarity"]
    )

    best = temp.sort_values("final_score", ascending=False).iloc[0]

    summary_rows.append({
        "profile_id": profile_id,
        "profile_name": profile["profile_name"],
        "preferred_breeds": ", ".join(profile.get("preferred_breeds", [])),
        "best_dog": best["Name"],
        "best_dog_breed1": best["breed1_label"],
        "best_dog_breed2": best["breed2_label"],
        "breed_score": best["breed_score"],
        "final_score": best["final_score"]
    })

summary = pd.DataFrame(summary_rows)
summary

,profile_id,profile_name,preferred_breeds,best_dog,best_dog_breed1,best_dog_breed2,breed_score,final_score
0,family_children_apartment,Famiglia con bambini in appartamento,"Labrador Retriever, Golden Retriever",Bobby,Golden Retriever,NaN,1.0,0.965444
1,single_active_person,Persona singola attiva con giardino,Rottweiler,Tony,Rottweiler,NaN,1.0,0.971217
2,senior_applicant,Richiedente over 60,,Sara,Poodle,Mixed Breed,1.0,0.969688
3,experienced_countryside_family,Famiglia esperta in casa di campagna,,Cookie,Mixed Breed,NaN,1.0,0.986828
4,first_time_adopter,Prima adozione senza esperienza,"Beagle, Cocker Spaniel",D21(090916) Beagle,Beagle,NaN,1.0,0.949195


## 11. Conclusioni

Il notebook permette di testare il sistema di matching su più profili famiglia.

L'aggiunta della razza rende il matching più realistico perché molte famiglie possono avere preferenze esplicite su una o più razze.

Il sistema assegna più peso alla razza principale e un peso parziale alla seconda razza, integrando questa informazione con:

- preferenze strutturate;
- vincoli hard e soft;
- similarità semantica tramite BERT.
